In [1]:
import os, gc, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)


In [2]:
!pip install -q dagshub mlflow optuna

import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment2', mlflow=True)

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

mlflow.set_experiment("XGBoost_Training")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 106.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7f8951d3-54cb-416d-9867-dd498954ad21&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=214017312acec20f0674f6915ffe9cf00cfece6d346352e46c4b48c3bf3a6ce7




Output()

Accessing as smama23

Initialized MLflow to track repo "smama23/MLassignment2"

Repository smama23/MLassignment2 initialized!

<Experiment: artifact_location='mlflow-artifacts:/010484efdcfe4e5aa8298a56cff32683', creation_time=1777835883396, experiment_id='3', last_update_time=1777835883396, lifecycle_stage='active', name='XGBoost_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [3]:
DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
train_transaction = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_identity = pd.read_csv(f"{DATA_DIR}/train_identity.csv")

df_raw = train_transaction.merge(train_identity, on="TransactionID", how="left")
print("Raw shape:", df_raw.shape)
print("Fraud ratio:", df_raw["isFraud"].mean())

del train_transaction, train_identity
gc.collect()


Raw shape: (590540, 434)
Fraud ratio: 0.03499000914417313


240

## Cleaning

In [4]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == "object" or str(col_type) == "category":
            continue
        c_min, c_max = df[col].min(), df[col].max()
        if str(col_type)[:3] == "int":
            if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        else:
            df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory: {start_mem:.1f} MB -> {end_mem:.1f} MB ({100*(start_mem-end_mem)/start_mem:.1f}% reduction)")
    return df


In [5]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    mem_before = df_raw.memory_usage(deep=True).sum() / 1024**2

    y = df_raw["isFraud"].astype(np.int8).copy()
    df = df_raw.drop(columns=["isFraud"]).copy()
    df = reduce_mem_usage(df)

    mem_after = df.memory_usage(deep=True).sum() / 1024**2

    mlflow.log_param("strategy", "downcast_dtypes_keep_NaN")
    mlflow.log_param("nan_handling", "preserved (XGBoost handles natively)")
    mlflow.log_metric("memory_before_MB", mem_before)
    mlflow.log_metric("memory_after_MB", mem_after)
    mlflow.log_metric("num_features", df.shape[1])
    mlflow.log_metric("num_samples", df.shape[0])
    mlflow.log_metric("num_object_cols", df.select_dtypes(include="object").shape[1])
    mlflow.log_metric("nan_ratio_mean", df.isna().mean().mean())

    print(f"After cleaning: {df.shape}, memory: {mem_after:.1f} MB")


Memory: 2509.5 MB -> 1602.7 MB (36.1% reduction)
After cleaning: (590540, 433), memory: 1602.7 MB
🏃 View run XGBoost_Cleaning at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/d94920f4f66444efa81e24688c505513
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


## Feature Engineering

In [6]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = ((X["TransactionDT"] // 3600) % 24).astype(np.int8)
        X["day"] = (X["TransactionDT"] // (3600 * 24)).astype(np.int32)
        X["dayofweek"] = (X["day"] % 7).astype(np.int8)
        X["log_amount"] = np.log1p(X["TransactionAmt"]).astype(np.float32)
        X["amt_decimal"] = ((X["TransactionAmt"] - X["TransactionAmt"].astype(np.int64)) * 1000).round().astype(np.int32)

        for col in ["P_emaildomain", "R_emaildomain"]:
            if col in X.columns:
                s = X[col].astype(str)
                X[col + "_suffix"] = s.str.split(".").str[-1]
                X[col + "_bin"] = s.str.split(".").str[0]

        if "DeviceInfo" in X.columns:
            X["DeviceInfo_family"] = (
                X["DeviceInfo"].astype(str).str.split("/").str[0].str.lower()
            )
        if "id_30" in X.columns:
            X["OS_family"] = X["id_30"].astype(str).str.split(" ").str[0].str.lower()
        if "id_31" in X.columns:
            X["browser_family"] = X["id_31"].astype(str).str.split(" ").str[0].str.lower()
        if "id_33" in X.columns:
            s = X["id_33"].astype(str)
            X["screen_w"] = pd.to_numeric(s.str.split("x").str[0], errors="coerce").astype(np.float32)
            X["screen_h"] = pd.to_numeric(s.str.split("x").str[1], errors="coerce").astype(np.float32)

        if "D1" in X.columns:
            X["D1n"] = (X["day"] - X["D1"].fillna(0)).astype(np.float32)

        X["uid_card1"] = X["card1"].astype(str)
        if "addr1" in X.columns:
            X["uid_card1_addr1"] = X["card1"].astype(str) + "_" + X["addr1"].astype(str)
        if "P_emaildomain" in X.columns and "addr1" in X.columns:
            X["uid_card1_addr1_pemail"] = (
                X["card1"].astype(str) + "_" + X["addr1"].astype(str) + "_" + X["P_emaildomain"].astype(str)
            )
        if "D1n" in X.columns and "addr1" in X.columns:
            X["uid_card1_addr1_d1n"] = (
                X["card1"].astype(str) + "_" + X["addr1"].astype(str) + "_" + X["D1n"].astype(str)
            )
        return X


class AggregateByUID(BaseEstimator, TransformerMixin):

    def __init__(self, uid_cols=None, agg_cols=None, agg_funcs=("mean", "std")):
        self.uid_cols = uid_cols or []
        self.agg_cols = agg_cols or []
        self.agg_funcs = list(agg_funcs)

    def fit(self, X, y=None):
        self.agg_maps_ = {}
        for uid in self.uid_cols:
            if uid not in X.columns:
                continue
            for col in self.agg_cols:
                if col not in X.columns:
                    continue
                grouped = X.groupby(uid)[col]
                for fn in self.agg_funcs:
                    self.agg_maps_[(uid, col, fn)] = grouped.agg(fn).astype(np.float32)
        return self

    def transform(self, X):
        X = X.copy()
        for (uid, col, fn), m in self.agg_maps_.items():
            X[f"{uid}__{col}__{fn}"] = X[uid].map(m).astype(np.float32)
        return X


class FrequencyCountEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, cols=None):
        self.cols = cols

    def fit(self, X, y=None):
        if self.cols is None:
            self.cols_ = X.select_dtypes(include="object").columns.tolist()
        else:
            self.cols_ = list(self.cols)
        self.count_maps_, self.freq_maps_ = {}, {}
        n = len(X)
        for col in self.cols_:
            if col not in X.columns:
                continue
            vc = X[col].astype(str).value_counts(dropna=False)
            self.count_maps_[col] = vc.astype(np.float32)
            self.freq_maps_[col] = (vc / n).astype(np.float32)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols_:
            if col not in X.columns:
                continue
            s = X[col].astype(str)
            X[col + "_count"] = s.map(self.count_maps_[col]).fillna(0).astype(np.float32)
            X[col] = s.map(self.freq_maps_[col]).fillna(0).astype(np.float32)
        return X


class ColumnPruner(BaseEstimator, TransformerMixin):

    def __init__(self, max_nan_ratio=0.95):
        self.max_nan_ratio = max_nan_ratio

    def fit(self, X, y=None):
        nan_ratio = X.isna().mean()
        nunique = X.nunique(dropna=False)
        self.cols_ = [c for c in X.columns
                      if nan_ratio[c] <= self.max_nan_ratio
                      and nunique[c] > 1
                      and X[c].dtype != "object"]
        return self

    def transform(self, X):
        return X.reindex(columns=self.cols_)


In [7]:
with mlflow.start_run(run_name="XGBoost_Feature_Engineering"):
    fe = FeatureEngineer()
    df_fe = fe.fit_transform(df)

    new_cols = [c for c in df_fe.columns if c not in df.columns]
    mlflow.log_param("time_features", "hour, day, dayofweek, log_amount, amt_decimal")
    mlflow.log_param("email_features", "P/R_emaildomain_suffix + _bin")
    mlflow.log_param("device_features", "DeviceInfo_family, OS_family, browser_family, screen_w/h")
    mlflow.log_param("uid_features", "card1, card1+addr1, +pemail, +D1n")
    mlflow.log_metric("num_features_after_fe", df_fe.shape[1])
    mlflow.log_metric("num_new_features", len(new_cols))
    print(f"Added {len(new_cols)} features. New shape: {df_fe.shape}")
    print("Sample new cols:", new_cols[:15])


Added 19 features. New shape: (590540, 452)
Sample new cols: ['hour', 'day', 'dayofweek', 'log_amount', 'amt_decimal', 'P_emaildomain_suffix', 'P_emaildomain_bin', 'R_emaildomain_suffix', 'R_emaildomain_bin', 'DeviceInfo_family', 'OS_family', 'browser_family', 'screen_w', 'screen_h', 'D1n']
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/8a347741d8584ffcb60e3f5067c3bfc0
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


In [8]:
with mlflow.start_run(run_name="XGBoost_Feature_Engineering_Aggregations"):
    UID_COLS = ["uid_card1", "uid_card1_addr1", "uid_card1_addr1_pemail", "uid_card1_addr1_d1n"]
    AGG_COLS = ["TransactionAmt", "D15", "id_02", "C13", "C14", "D4"]
    AGG_FUNCS = ("mean", "std")

    agg = AggregateByUID(uid_cols=UID_COLS, agg_cols=AGG_COLS, agg_funcs=AGG_FUNCS)
    df_agg = agg.fit_transform(df_fe)

    mlflow.log_param("uid_cols", ", ".join(UID_COLS))
    mlflow.log_param("agg_cols", ", ".join(AGG_COLS))
    mlflow.log_param("agg_funcs", ", ".join(AGG_FUNCS))
    mlflow.log_metric("num_aggregation_features", len(agg.agg_maps_))
    mlflow.log_metric("num_features_after_agg", df_agg.shape[1])
    print(f"Aggregations added: {len(agg.agg_maps_)}. Shape: {df_agg.shape}")


Aggregations added: 48. Shape: (590540, 500)
🏃 View run XGBoost_Feature_Engineering_Aggregations at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/af13aedc58884657b1397b1500faedf7
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


## Feature Selection

In [9]:
with mlflow.start_run(run_name="XGBoost_Feature_Selection_v1"):
    enc = FrequencyCountEncoder()
    df_enc = enc.fit_transform(df_agg)

    pruner = ColumnPruner(max_nan_ratio=0.95)
    df_fs = pruner.fit_transform(df_enc)

    mlflow.log_param("encoding", "frequency + count for all object cols")
    mlflow.log_param("pruning_rule", "drop nunique<=1 OR nan_ratio>0.95 OR dtype==object")
    mlflow.log_metric("num_categorical_encoded", len(enc.cols_))
    mlflow.log_metric("num_features_after_encoding", df_enc.shape[1])
    mlflow.log_metric("num_features_after_prune", df_fs.shape[1])

    print(f"Encoded {len(enc.cols_)} cols. Final feature count: {df_fs.shape[1]}")
    print("Dtypes:", df_fs.dtypes.value_counts().to_dict())


Encoded 42 cols. Final feature count: 535
Dtypes: {dtype('float32'): 528, dtype('int32'): 4, dtype('int8'): 2, dtype('int16'): 1}
🏃 View run XGBoost_Feature_Selection_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/950d26e6a27a437b99bcb31f03a9d3bf
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


In [10]:
with mlflow.start_run(run_name="XGBoost_Feature_Selection_Importance"):
    cutoff = df["TransactionDT"].quantile(0.8)
    train_mask = df["TransactionDT"] < cutoff
    val_mask = ~train_mask

    quick = XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", eval_metric="auc", random_state=42,
        early_stopping_rounds=30,
    )
    quick.fit(df_fs[train_mask], y[train_mask],
              eval_set=[(df_fs[val_mask], y[val_mask])], verbose=False)

    imp = pd.Series(quick.feature_importances_, index=df_fs.columns).sort_values(ascending=False)
    keep_imp = imp[imp > 0].index.tolist()

    mlflow.log_param("importance_cutoff", "> 0")
    mlflow.log_metric("num_features_kept_by_importance", len(keep_imp))
    mlflow.log_metric("num_features_dropped", df_fs.shape[1] - len(keep_imp))
    mlflow.log_metric("quick_val_auc", roc_auc_score(y[val_mask], quick.predict_proba(df_fs[val_mask])[:, 1]))

    print(f"Importance > 0: kept {len(keep_imp)} / {df_fs.shape[1]}")
    print("Top 15:", imp.head(15).to_dict())

USE_IMPORTANCE_PRUNED = False
df_train = df_fs[keep_imp] if USE_IMPORTANCE_PRUNED else df_fs
print("Training matrix:", df_train.shape)


Importance > 0: kept 499 / 535
Top 15: {'V258': 0.21756476163864136, 'V149': 0.051927242428064346, 'V294': 0.03602232411503792, 'C7': 0.027805890887975693, 'V70': 0.02216249145567417, 'V91': 0.01647014170885086, 'V156': 0.016442008316516876, 'C4': 0.01552759762853384, 'V283': 0.009901844896376133, 'C14': 0.009884977713227272, 'addr2': 0.009279882535338402, 'V205': 0.009161190129816532, 'V154': 0.008772480301558971, 'C8': 0.008724135346710682, 'C13': 0.008173280395567417}
🏃 View run XGBoost_Feature_Selection_Importance at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/45ec856ab3c44ea298a77f6ac2ad14c5
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
Training matrix: (590540, 535)


## Training

In [11]:
cutoff = df["TransactionDT"].quantile(0.8)
train_mask = df["TransactionDT"] < cutoff
val_mask = ~train_mask

X_tr, X_vl = df_train[train_mask], df_train[val_mask]
y_tr, y_vl = y[train_mask], y[val_mask]
print(f"Train: {X_tr.shape}, Val: {X_vl.shape}")
print(f"Fraud rate — train: {y_tr.mean():.4f}, val: {y_vl.mean():.4f}")


Train: (472432, 535), Val: (118108, 535)
Fraud rate — train: 0.0351, val: 0.0344


In [12]:
with mlflow.start_run(run_name="XGBoost_Training_Baseline"):
    params = dict(
        n_estimators=2000,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.5,
        min_child_weight=1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="auc",
        random_state=42,
        early_stopping_rounds=100,
    )
    model = XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=200)

    train_auc = roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1])
    val_auc = roc_auc_score(y_vl, model.predict_proba(X_vl)[:, 1])

    for k, v in params.items():
        mlflow.log_param(k, v)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("best_iteration", model.best_iteration)
    print(f"Baseline — train: {train_auc:.5f}, val: {val_auc:.5f}, best_iter: {model.best_iteration}")


[0]	validation_0-auc:0.80225
[200]	validation_0-auc:0.92864
[400]	validation_0-auc:0.93517
[600]	validation_0-auc:0.93773
[798]	validation_0-auc:0.93795
Baseline — train: 0.99879, val: 0.93807, best_iter: 698
🏃 View run XGBoost_Training_Baseline at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/4ffa6dcf64f94b5a9c5f096e72446bc2
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


In [13]:
with mlflow.start_run(run_name="XGBoost_Training_CrossValidation"):
    n_splits = 3
    sorted_idx = df["TransactionDT"].sort_values().index
    fold_size = len(sorted_idx) // (n_splits + 1)

    fold_aucs = []
    for fold in range(n_splits):
        tr_idx = sorted_idx[: fold_size * (fold + 1)]
        vl_idx = sorted_idx[fold_size * (fold + 1): fold_size * (fold + 2)]

        m = XGBClassifier(
            n_estimators=1500, max_depth=8, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.5,
            tree_method="hist", eval_metric="auc",
            random_state=42, early_stopping_rounds=80,
        )
        m.fit(df_train.loc[tr_idx], y.loc[tr_idx],
              eval_set=[(df_train.loc[vl_idx], y.loc[vl_idx])], verbose=False)
        auc = roc_auc_score(y.loc[vl_idx], m.predict_proba(df_train.loc[vl_idx])[:, 1])
        fold_aucs.append(auc)
        mlflow.log_metric(f"fold_{fold}_auc", auc)
        print(f"  fold {fold}: {auc:.5f}  (best_iter={m.best_iteration})")

    mlflow.log_metric("cv_mean_auc", float(np.mean(fold_aucs)))
    mlflow.log_metric("cv_std_auc", float(np.std(fold_aucs)))
    mlflow.log_param("cv_strategy", "expanding-window time-based 3-fold")
    print(f"CV mean: {np.mean(fold_aucs):.5f}  std: {np.std(fold_aucs):.5f}")


  fold 0: 0.91641  (best_iter=211)
  fold 1: 0.93369  (best_iter=342)
  fold 2: 0.93667  (best_iter=417)
CV mean: 0.92893  std: 0.00893
🏃 View run XGBoost_Training_CrossValidation at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/c8cef373c2fd4ca9b47b2464b3c541b2
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


In [16]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

with mlflow.start_run(run_name="XGBoost_Training_Optuna") as parent_run:
    def objective(trial):
        params = {
            "n_estimators": 800,
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 100),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "tree_method": "hist",
            "device": "cuda",
            "eval_metric": "auc",
            "random_state": 42,
            "early_stopping_rounds": 30,
        }
        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            for k, v in params.items():
                mlflow.log_param(k, v)
            m = XGBClassifier(**params)
            m.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
            auc = roc_auc_score(y_vl, m.predict_proba(X_vl)[:, 1])
            mlflow.log_metric("val_auc", auc)
            mlflow.log_metric("best_iteration", m.best_iteration)
        return auc

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=25, show_progress_bar=False)

    best = study.best_trial
    for k, v in best.params.items():
        mlflow.log_param(f"best_{k}", v)
    mlflow.log_metric("best_val_auc", best.value)
    mlflow.log_metric("n_trials", len(study.trials))
    print(f"Best AUC: {best.value:.5f}")
    print(f"Best params: {best.params}")

    BEST_PARAMS = best.params


🏃 View run trial_0 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/17d6fff25a5d41e98cae18b6e3613be8
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
🏃 View run trial_1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/7ce5c02d0c404c00b5b9cf788e78c2f3
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
🏃 View run trial_2 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/da93dbe3aec04bf4a030b1b46a6991b5
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
🏃 View run trial_3 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/34bbffff48474781aa68aefd50815371
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
🏃 View run trial_4 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/cffe0517e4d349758f9c7142fa8abc2d
🧪 View experiment at: 

In [17]:
with mlflow.start_run(run_name="XGBoost_Training_Final"):
    final_params = {
        **BEST_PARAMS,
        "n_estimators": 2000,
        "tree_method": "hist",
        "device": "cuda",
        "eval_metric": "auc",
        "random_state": 42,
        "early_stopping_rounds": 80,
    }
    model_final = XGBClassifier(**final_params)
    model_final.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=200)

    train_auc = roc_auc_score(y_tr, model_final.predict_proba(X_tr)[:, 1])
    val_auc = roc_auc_score(y_vl, model_final.predict_proba(X_vl)[:, 1])

    for k, v in final_params.items():
        mlflow.log_param(k, v)
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("best_iteration", model_final.best_iteration)
    BEST_N_ESTIMATORS = int(model_final.best_iteration) + 1
    print(f"Final — train: {train_auc:.5f}, val: {val_auc:.5f}, best_iter: {model_final.best_iteration}")


[0]	validation_0-auc:0.80090
[200]	validation_0-auc:0.93057
[400]	validation_0-auc:0.93868
[600]	validation_0-auc:0.94112
[800]	validation_0-auc:0.94148
[823]	validation_0-auc:0.94151
Final — train: 0.99780, val: 0.94151, best_iter: 743
🏃 View run XGBoost_Training_Final at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/87fd77f563f24868ab97fc40367688a4
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3


In [18]:
UID_COLS = ["uid_card1", "uid_card1_addr1", "uid_card1_addr1_pemail", "uid_card1_addr1_d1n"]
AGG_COLS = ["TransactionAmt", "D15", "id_02", "C13", "C14", "D4"]

pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("aggregations", AggregateByUID(uid_cols=UID_COLS, agg_cols=AGG_COLS, agg_funcs=("mean", "std"))),
    ("encoding", FrequencyCountEncoder()),
    ("pruner", ColumnPruner(max_nan_ratio=0.95)),
    ("model", XGBClassifier(
        **{k: v for k, v in BEST_PARAMS.items() if k != "n_estimators"},
        n_estimators=BEST_N_ESTIMATORS,
        tree_method="hist",
        device="cuda",
        eval_metric="auc",
        random_state=42,
    )),
])
print(pipeline)


Pipeline(steps=[('feature_engineering', FeatureEngineer()),
                ('aggregations',
                 AggregateByUID(agg_cols=['TransactionAmt', 'D15', 'id_02',
                                          'C13', 'C14', 'D4'],
                                agg_funcs=['mean', 'std'],
                                uid_cols=['uid_card1', 'uid_card1_addr1',
                                          'uid_card1_addr1_pemail',
                                          'uid_card1_addr1_d1n'])),
                ('encoding', FrequencyCountEncoder()),
                ('pruner', ColumnPruner()),
                ('model',
                 XGBClassifier(b...
                               gamma=3.100308334813918, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.02154533371662402, max_bin=None,
                               max_cat_threshold=None, max_cat_to_oneh

In [19]:
with mlflow.start_run(run_name="XGBoost_Pipeline_Final"):
    X_full = df_raw.drop(columns=["isFraud"])
    y_full = df_raw["isFraud"].astype(np.int8)

    cutoff_full = X_full["TransactionDT"].quantile(0.8)
    tr = X_full["TransactionDT"] < cutoff_full
    vl = ~tr

    pipeline.fit(X_full[tr], y_full[tr])

    val_pred = pipeline.predict_proba(X_full[vl])[:, 1]
    val_auc = roc_auc_score(y_full[vl], val_pred)
    train_pred = pipeline.predict_proba(X_full[tr])[:, 1]
    train_auc = roc_auc_score(y_full[tr], train_pred)

    for k, v in BEST_PARAMS.items():
        mlflow.log_param(f"xgb_{k}", v)
    mlflow.log_param("pipeline_steps", " -> ".join(name for name, _ in pipeline.steps))
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    print(f"Pipeline (fit on train, val held out) — train: {train_auc:.5f}, val: {val_auc:.5f}")

    pipeline.fit(X_full, y_full)

    input_example = X_full.head(5)
    signature = infer_signature(input_example, pipeline.predict_proba(input_example)[:, 1])

    mlflow.sklearn.log_model(
        sk_model=pipeline,
        name="xgboost_pipeline",
        signature=signature,
        input_example=input_example,
    )
    mlflow.log_param("refit_on", "full_train")
    print("Pipeline logged to MLflow.")
    print("Run ID:", mlflow.active_run().info.run_id)


Pipeline (fit on train, val held out) — train: 0.99761, val: 0.93480


2026/05/04 14:05:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Pipeline logged to MLflow.
Run ID: fdf724c132be4733bd33e8eeb14b2edc
🏃 View run XGBoost_Pipeline_Final at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3/runs/fdf724c132be4733bd33e8eeb14b2edc
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/3
